# Setup

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on -DLLAVA_BUILD=off" pip install llama-cpp-python --upgrade --force-reinstall --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfully uninstalled MarkupS

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
import os
import time
import re
import pandas as pd
from llama_cpp import Llama
from typing import Optional

In [40]:
!wget -P /content/models https://huggingface.co/bartowski/Qwen_Qwen3-8B-GGUF/resolve/main/Qwen_Qwen3-8B-Q5_K_M.gguf

--2026-03-15 12:23:03--  https://huggingface.co/bartowski/Qwen_Qwen3-8B-GGUF/resolve/main/Qwen_Qwen3-8B-Q5_K_M.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.97, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/680f7b70e445a4be6af508b2/c45b5e46a5c4d7d14b2e2f1763fb3b808c5aa82ac1c08195ca1e574d494cda76?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Qwen_Qwen3-8B-Q5_K_M.gguf%3B+filename%3D%22Qwen_Qwen3-8B-Q5_K_M.gguf%22%3B&Expires=1773580983&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzczNTgwOTgzfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjgwZjdiNzBlNDQ1YTRiZTZhZjUwOGIyL2M0NWI1ZTQ2YTVjNGQ3ZDE0YjJlMmYxNzYzZmIzYjgwOGM1YWE4MmFjMWMwODE5NWNhMWU1NzRkNDk0Y2RhNzZcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=ueoSGfruRWuSgFZyvt-0zd

# Code

In [2]:
# --------------------
# DS Preparation Model
# --------------------
class LlamaCppDatasetPreparationModelSettings:
    N_CTX: int = 6000
    N_GPU_LAYERS: int = -1
    N_BATCH: int = 2048
    N_THREADS: int = 4
    N_THREADS_BATCH: int = 4
    VERBOSE: bool = False
    FLASH_ATTN: bool = True


class LlamaCppDatasetPreparationInferenceSettings:
    MAX_TOKENS: int = 4096    # ACCOUNT FOR REASONING
    TEMPERATURE: float = 0.7
    PRESENCE_PENALTY: float = 0.0
    REPEAT_PENALTY: float = 1.1
    TOP_K: int = 20
    TOP_P: float = 0.95

In [3]:
SQL_EXPLAINER_SYSTEM_PROMPT_DS_PREP = """You are an expert SQL analyst. When the user provides a SQL query, respond with a single concise paragraph explaining how the query works, covering what data it retrieves, the tables or sources involved, any filtering conditions, joins, groupings, or ordering applied, and the expected output. Follow these rules strictly:

1. Column names: Treat any phrase between SELECT and FROM, or in a WHERE clause, as a single column name — even if it contains slashes (/), spaces, parentheses, or the word "as". Do not split column names into separate entities or interpret embedded words as SQL keywords. For example, "School/Club Team" is one column, and "Entered office as Head of State or Government" is one column name, not a column with an alias.
2. AS aliases: Only treat something as a SQL alias if the keyword AS appears explicitly between the SELECT column and the output column name, outside of what appears to be part of the column name itself.
3. COUNT syntax: If COUNT is used without parentheses (e.g. COUNT No.), flag this precisely as a missing-parentheses syntax error. Note that COUNT(column) is valid; only COUNT without parentheses is invalid.
4. String filter values: Treat filter values like '2005-06' or '1996-97' as string literals (e.g. season labels), not numeric ranges or date arithmetic, unless the column type explicitly suggests otherwise.
5. Multi-row output: Never state that a query returns only the "first" matching row unless a LIMIT, TOP, or ROWNUM clause is present. Without such a clause, all matching rows are returned.
6. Interpret column names literally: Do not use the text of a column name to infer or narrate what the data means in the real world. Describe what the query retrieves structurally, not what the column name implies about the domain.

Do not use bullet points, headings, or code blocks in your response. Keep the explanation clear and accessible, as if explaining to a developer who understands SQL but wants a quick summary. Do not restate the query in your response."""


class PromptManager:
    """Utility class to construct prompts using a chat-style template."""

    def __init__(self, system: Optional[str] = None):
        """Initializes the PromptManager."""
        self.SYSTEM_PROMPT = system

    def get_prompt(
        self,
        query: str,
        output: Optional[str] = None
    ) -> str:
        """Construct a formatted prompt for the language model."""
        prompt = ""

        if self.SYSTEM_PROMPT:
            prompt += f"""<|im_start|>system\n{self.SYSTEM_PROMPT}<|im_end|>\n"""

        prompt += f"""<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"""

        if output:
            prompt += f"""{output}<|im_end|>"""

        return prompt

    def get_prompt_messages(
        self,
        query: str,
        output: Optional[str] = None
    ) -> list[dict]:
        """Construct a list of chat messages in OpenAI-style format."""
        messages = []

        if self.SYSTEM_PROMPT:
            messages.append({"role": "system", "content": self.SYSTEM_PROMPT})
        messages.append({"role": "user", "content": query})
        if output:
            messages.append({"role": "assistant", "content": output})

        return messages

In [4]:
def separate_thinking_and_response(response: str) -> tuple[str | None, str]:
    """
    Separates the thinking part from the actual response.

    Returns a tuple of (thinking, actual) where:
    - thinking: content inside <think>...</think> tags, or None if not present
    - actual: the response with thinking removed and code fences stripped
    """
    thinking = None

    if '</think>' in response:
        think_start = response.find('<think>')
        think_end = response.find('</think>') + len('</think>')

        if think_start != -1:
            thinking = response[think_start + len('<think>'):think_end - len('</think>')].strip()

        response = response[think_end:].strip()

    actual = re.sub(r"```(?:json|xml)?\s*([\s\S]*?)```", r"\1", response, flags=re.IGNORECASE).strip()

    return thinking, actual

In [5]:
OUTPUT_DF_DICT_TEMPLATE = {
    "system": [],
    "query": [],
    "explanation": [],
    "reasoning": []
}

INPUT_DATASET_FILEPATH = "/content/drive/MyDrive/dataset_incomplete.csv"
INPUT_DATASET_SYSTEM_COLUMN = "system"
INPUT_DATASET_QUERY_COLUMN = "query"
INPUT_DATASET_EXPLANATION_COLUMN = "explanation"
INPUT_DATASET_REASONING_COLUMN = "reasoning"

OUTPUT_DATASET_FILEPATH = "/content/drive/MyDrive/dataset_complete.csv"

GGUF_MODEL_FILEPATH = "/content/models/Qwen_Qwen3-8B-Q5_K_M.gguf"

output_df_dict = OUTPUT_DF_DICT_TEMPLATE.copy()

In [6]:
model_loading_start_time = time.time()
print(f"Loading model '{GGUF_MODEL_FILEPATH}' ...")
model = Llama(
    model_path=GGUF_MODEL_FILEPATH,
    n_ctx=LlamaCppDatasetPreparationModelSettings.N_CTX,
    n_gpu_layers=LlamaCppDatasetPreparationModelSettings.N_GPU_LAYERS,
    n_batch=LlamaCppDatasetPreparationModelSettings.N_BATCH,
    n_threads=LlamaCppDatasetPreparationModelSettings.N_THREADS,
    n_threads_batch=LlamaCppDatasetPreparationModelSettings.N_THREADS_BATCH,
    verbose=LlamaCppDatasetPreparationModelSettings.VERBOSE,
    flash_attn=LlamaCppDatasetPreparationModelSettings.FLASH_ATTN
)
print(f"Model '{GGUF_MODEL_FILEPATH}' loaded successfully in {time.time() - model_loading_start_time:.2f} seconds")

Loading model '/content/models/Qwen_Qwen3-8B-Q5_K_M.gguf' ...


llama_context: n_ctx_per_seq (6000) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model '/content/models/Qwen_Qwen3-8B-Q5_K_M.gguf' loaded successfully in 2.42 seconds


In [7]:
prompt_manager = PromptManager(SQL_EXPLAINER_SYSTEM_PROMPT_DS_PREP)

input_df = pd.read_csv(INPUT_DATASET_FILEPATH)

In [8]:
# Resume from checkpoint if output file already exists
if os.path.exists(OUTPUT_DATASET_FILEPATH):
    existing_df = pd.read_csv(OUTPUT_DATASET_FILEPATH)
    rows_already_processed = len(existing_df)
    output_df_dict = existing_df.to_dict(orient="list")
    input_df = input_df.iloc[rows_already_processed:]
    print(f"Resuming from row {rows_already_processed + 1} ({len(input_df)} rows remaining) ...")
else:
    rows_already_processed = 0

In [9]:
total_processing_start_time = time.time()
print(f"Performing dataset preparation on {len(input_df)} rows ...")
for row_idx, row in input_df.iterrows():
    row_processing_start_time = time.time()
    print(f"\n\nProcessing row ({row_idx + 1} / {len(input_df)}) ...\n\n")

    output_df_dict[INPUT_DATASET_SYSTEM_COLUMN].append(row[INPUT_DATASET_SYSTEM_COLUMN])
    output_df_dict[INPUT_DATASET_QUERY_COLUMN].append(row[INPUT_DATASET_QUERY_COLUMN])

    try:
        row_query = row[INPUT_DATASET_QUERY_COLUMN]
        messages = prompt_manager.get_prompt_messages(query=row_query)
        print(f"QUERY:- {row_query}")

        response = model.create_chat_completion(
            messages=messages,
            max_tokens=LlamaCppDatasetPreparationInferenceSettings.MAX_TOKENS,
            temperature=LlamaCppDatasetPreparationInferenceSettings.TEMPERATURE,
            presence_penalty=LlamaCppDatasetPreparationInferenceSettings.PRESENCE_PENALTY,
            repeat_penalty=LlamaCppDatasetPreparationInferenceSettings.REPEAT_PENALTY,
            top_k=LlamaCppDatasetPreparationInferenceSettings.TOP_K,
            top_p=LlamaCppDatasetPreparationInferenceSettings.TOP_P,
        )
        response_text = response["choices"][0]["message"]["content"]
        reasoning, explanation = separate_thinking_and_response(response_text)
        usage = response["usage"]
        print(explanation)
        print(usage)

        # Saving output df after every row
        output_df_dict[INPUT_DATASET_EXPLANATION_COLUMN].append(explanation)
        output_df_dict[INPUT_DATASET_REASONING_COLUMN].append(reasoning)
        output_df = pd.DataFrame(output_df_dict)
        output_df.to_csv(OUTPUT_DATASET_FILEPATH, index=False)

        print(f"Row {row_idx + 1} processed in {time.time() - row_processing_start_time:.2f} seconds")
    except Exception as e:
        print(f"[WARNING] Failed to process row {row_idx + 1}! {str(e)}")

        # Saving output df after every row
        output_df_dict[INPUT_DATASET_EXPLANATION_COLUMN].append(None)
        output_df_dict[INPUT_DATASET_REASONING_COLUMN].append(None)
        output_df = pd.DataFrame(output_df_dict)
        output_df.to_csv(OUTPUT_DATASET_FILEPATH, index=False)


print(f"Dataset preparation for {len(input_df)} rows finished in {time.time() - total_processing_start_time:.2f} seconds")

Performing dataset preparation on 79703 rows ...


Processing row (1 / 79703) ...


QUERY:- SELECT Position FROM table WHERE School/Club Team = Butler CC (KS)
The query retrieves the "Position" column from the specified table where the value in the "School/Club Team" column exactly matches the string 'Butler CC (KS)'. It does not involve any joins, aggregations, or sorting, and returns all rows that meet this condition. The output consists of the positions associated with the team named 'Butler CC (KS)' as stored in the table's "Position" column.
{'prompt_tokens': 463, 'completion_tokens': 287, 'total_tokens': 750}
Row 1 processed in 12.33 seconds


Processing row (2 / 79703) ...


QUERY:- SELECT COUNT School/Club Team FROM table WHERE No. = 3
The query attempts to count the number of rows in the specified table where the column "No." equals 3, but it contains a syntax error due to the missing parentheses in the COUNT function. The column "School/Club Team" is referenced in the SELECT 

KeyboardInterrupt: 